In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Iterable, Iterator, List, Optional, Sequence, Tuple, Union

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

import xgboost as xgb
import lightgbm as lgb
from lightgbm import LGBMClassifier
import shap

from src.data_prepatation import create_train_val_test_splits
from src.apnea_feature_tiers import get_features, filter_df, TIER_1, TIER_2, TIER_3

from src.data_prepatation import create_train_val_test_splits

### Configuration

In [ ]:
from config import dir_config, random_seed
compiled_dir = dir_config.data.compiled
processed_dir = dir_config.data.processed

In [ ]:
target_future_step = 2

features = 'all' # options: 'all', 'tier1', 'tier12', 'tier123'
# features = 'tier1'
# features = 'tier12'
# features = 'tier123'


### Load Data

In [ ]:

# get all parquet files in the processed directory
parquet_files = list(Path(processed_dir).glob('*_with_metadata.parquet'))

# remove "agg_data_with_metadata.parquet" from the list of parquet files
parquet_files = [f for f in parquet_files if f.name != "agg_data_with_metadata.parquet"]

metadata = pd.read_csv(Path(processed_dir) / "participant_info.csv")

# # find sid which have ahi < 5
# non_apnea_sids = metadata[metadata['ahi'] < 5]['sid'].tolist()

# # remove parquet files that correspond to non-apnea sids in their name (eg "{sid}_with_metadata.parquet")
# parquet_files = [f for f in parquet_files if not any(sid in f.name for sid in non_apnea_sids)]

### Data processing

In [ ]:
train_X, train_y, fold_indices, val_X, val_y, test_X, test_y = create_train_val_test_splits(
    parquet_files,
    metadata=metadata,
    top_features=None,
    top_features_lag=0,
    target_type='apnea',
    target_future_steps=target_future_step,
    val_ratio = 0.2,
    test_ratio=0.2,
    n_splits=5,
    random_seed=random_seed,
)

In [ ]:
def verify_splits(
    train_X, train_y, val_X, val_y, test_X, test_y,
    fold_indices, metadata: pd.DataFrame,
):
    severity_lookup = metadata.set_index('subject_id')['ahi_stratum']

    # --- 1. No subject leakage across splits ---
    train_subjects = set(train_X['subject_id'].unique())
    val_subjects   = set(val_X['subject_id'].unique())
    test_subjects  = set(test_X['subject_id'].unique())

    assert train_subjects.isdisjoint(val_subjects),  "LEAK: subjects in both train and val"
    assert train_subjects.isdisjoint(test_subjects), "LEAK: subjects in both train and test"
    assert val_subjects.isdisjoint(test_subjects),   "LEAK: subjects in both val and test"
    print("✓ No subject leakage across train/val/test")

    # --- 2. All subjects accounted for ---
    all_subjects = train_subjects | val_subjects | test_subjects
    expected     = set(metadata['subject_id'].unique())
    missing      = expected - all_subjects
    assert not missing, f"Missing subjects: {missing}"
    print(f"✓ All {len(all_subjects)} subjects accounted for")

    # --- 3. Split size ratios are roughly correct ---
    total = len(train_X) + len(val_X) + len(test_X)
    print(f"\nRow counts:  train={len(train_X)} ({len(train_X)/total:.0%})  "
          f"val={len(val_X)} ({len(val_X)/total:.0%})  "
          f"test={len(test_X)} ({len(test_X)/total:.0%})")
    print(f"Subject counts: train={len(train_subjects)}  "
          f"val={len(val_subjects)}  test={len(test_subjects)}")

    # --- 4. Severity distribution is balanced across splits ---
    print("\nSeverity distribution (% of subjects per split):")
    for name, subjects in [('train', train_subjects), ('val', val_subjects), ('test', test_subjects)]:
        dist = severity_lookup[list(subjects)].value_counts(normalize=True).sort_index()
        print(f"  {name:5s}: { {k: f'{v:.0%}' for k, v in dist.items()} }")

    # --- 5. Target distribution is balanced ---
    print(f"\nTarget mean:  train={train_y.mean():.3f}  "
          f"val={val_y.mean():.3f}  test={test_y.mean():.3f}")

    # --- 6. CV fold integrity ---
    print(f"\nCV folds ({len(fold_indices)} folds):")
    for i, (train_idx, val_idx) in enumerate(fold_indices):
        overlap = set(train_idx) & set(val_idx)
        assert not overlap, f"Fold {i}: train/val index overlap"
        fold_train_subs = set(train_X.iloc[train_idx]['subject_id'].unique())
        fold_val_subs   = set(train_X.iloc[val_idx]['subject_id'].unique())
        assert fold_train_subs.isdisjoint(fold_val_subs), f"Fold {i}: subject leakage"
        fold_val_dist = severity_lookup[list(fold_val_subs)].value_counts().sort_index().to_dict()
        print(f"  fold {i}: train={len(train_idx)} rows  val={len(val_idx)} rows  "
              f"val severity={fold_val_dist}")
    print("✓ No leakage in any CV fold")

verify_splits(train_X, train_y, val_X, val_y, test_X, test_y, fold_indices, metadata)

Keep only tier 1 and tier 2 feature

In [ ]:
# remove items containing "_range" or "_slope"
TIER_1 = [feature for feature in TIER_1 if "_range" not in feature]
TIER_2 = [feature for feature in TIER_2 if "_range" not in feature]
TIER_3 = [feature for feature in TIER_3 if "_range" not in feature]

if features == 'all':
    columns_to_keep = train_X.columns.tolist()
elif features == 'tier1':
    columns_to_keep = TIER_1
elif features == 'tier12':
    columns_to_keep = TIER_1 + TIER_2
elif features == 'tier123':
    columns_to_keep = TIER_1 + TIER_2 + TIER_3
else:
    raise ValueError(f"Invalid value for features: {features}")

# delete all columns which has "_range" or "_slope" in their name from columns_to_keep
# columns_to_keep = [col for col in columns_to_keep if "_range" not in col and "_slope" not in col]

train_X = train_X[[col for col in train_X.columns if col in columns_to_keep]]
val_X = val_X[[col for col in val_X.columns if col in columns_to_keep]]
test_X = test_X[[col for col in test_X.columns if col in columns_to_keep]]

In [ ]:
train_X.shape, train_y.shape, val_X.shape, val_y.shape, test_X.shape, test_y.shape

### Check data

In [ ]:
import importlib
import src.eda_functions
importlib.reload(src.eda_functions)

from src.eda_functions import run_full_eda, get_redundant_feature_groups

scores_df, metadata_features, timeseries_features = run_full_eda(train_X, train_y, metadata)

In [ ]:
def select_features(
    train_X: pd.DataFrame,
    val_X: pd.DataFrame,
    test_X: pd.DataFrame,
    timeseries_features: list[str],
    scores_df: pd.DataFrame,
    correlation_threshold: float = 0.9,
    nzv_threshold: float = 0.01,
    nzv_univariate_quantile: float = 0.25,
    keep_overrides: dict[str, str] | None = None,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, list[str]]:
    keep_overrides = keep_overrides or {
        'ibi_std':   'ibi_rmssd',
        'snore_std': 'snore_min',
        'sao2_mean': 'sao2_min',
    }

    # ── Step 1: Redundancy removal ─────────────────────────────────────────────
    print("=" * 60)
    print("STEP 1 — Redundancy Removal")
    print("=" * 60)
    redundant_groups = get_redundant_feature_groups(
        train_X, timeseries_features, correlation_threshold
    )
    features_to_drop = [feat for group in redundant_groups for feat in group[1:]]

    overrides_applied = []
    for auto_kept, keep_instead in keep_overrides.items():
        if keep_instead in features_to_drop and auto_kept not in features_to_drop:
            features_to_drop.remove(keep_instead)
            features_to_drop.append(auto_kept)
            overrides_applied.append(
                f"  swapped: keep={keep_instead:30s}  drop={auto_kept} (domain override)"
            )

    if overrides_applied:
        print("\nDomain knowledge overrides applied:")
        for msg in overrides_applied:
            print(msg)

    features_to_drop = list(set(features_to_drop))
    print(f"\nFeatures dropped (redundancy): {len(features_to_drop)}")

    # ── Step 2: Near-zero variance recheck ────────────────────────────────────
    print("\n" + "=" * 60)
    print("STEP 2 — Near-Zero Variance Recheck")
    print("=" * 60)
    remaining_ts = [f for f in timeseries_features if f not in features_to_drop]

    numeric = train_X[remaining_ts].select_dtypes(include=np.number)
    std = numeric.std()
    near_zero = std[std < nzv_threshold].sort_values()

    if near_zero.empty:
        print("✓ No near-zero variance features remaining.")
        nzv_to_drop = []
    else:
        print(f"⚠ {len(near_zero)} near-zero variance features remaining:")
        print(near_zero.to_string())
        bottom_quantile = scores_df.tail(
            int(len(scores_df) * nzv_univariate_quantile)
        )['feature'].tolist()
        nzv_to_drop = [f for f in near_zero.index if f in bottom_quantile]
        nzv_keep    = [f for f in near_zero.index if f not in nzv_to_drop]
        print(f"\nAuto-dropping (near-zero AND bottom {nzv_univariate_quantile:.0%} univariate rank): {nzv_to_drop}")
        print(f"Keeping  (near-zero but has univariate signal):            {nzv_keep}")

    # ── Step 3: Apply all drops and report ────────────────────────────────────
    print("\n" + "=" * 60)
    print("STEP 3 — Final Feature Set")
    print("=" * 60)
    all_features_to_drop = list(set(features_to_drop + nzv_to_drop))

    train_X_clean = train_X.drop(columns=all_features_to_drop, errors='ignore')
    val_X_clean   = val_X.drop(columns=all_features_to_drop, errors='ignore')
    test_X_clean  = test_X.drop(columns=all_features_to_drop, errors='ignore')

    final_features = [
        c for c in train_X_clean.columns
        if c not in ['subject_id', 'target']
        and c in timeseries_features  # exclude metadata from count
    ]

    print(f"Started with:            {len(timeseries_features)} time-series features")
    print(f"Dropped (redundancy):    {len(features_to_drop)}")
    print(f"Dropped (near-zero NZV): {len(nzv_to_drop)}")
    print(f"Remaining features:      {len(final_features)}")

    surviving_scores = scores_df[scores_df['feature'].isin(final_features)]
    print(f"\nTop 20 surviving features by combined rank:")
    print(surviving_scores.head(20)[
        ['feature', 'abs_correlation', 'mutual_info', 'combined_rank']
    ].to_string(index=False))

    return train_X_clean, val_X_clean, test_X_clean, final_features

In [ ]:
def get_top_features_for_lags(
    scores_df: pd.DataFrame,
    final_features: list[str],
    top_n: int = 10,
) -> list[str]:
    """
    Return top N time-series features ranked by combined univariate score,
    filtered to only features that survived selection. These are the
    candidates for lag feature generation.

    Args:
        scores_df:      Output of compute_univariate_scores().
        final_features: Output of select_features() — only surviving features.
        top_n:          Number of top features to return.

    Returns:
        List of top N feature names ranked by combined_rank.
    """
    surviving = scores_df[scores_df['feature'].isin(final_features)]
    top = surviving.head(top_n)

    print(f"Top {top_n} features for lag generation (ranked by combined score):")
    print(top[['feature', 'abs_correlation', 'mutual_info', 'combined_rank']].to_string(index=False))

    return top['feature'].tolist()

In [ ]:
# plot bar graph of

## Add lag features

In [ ]:
with_metadata = True
with_pca = False
max_feature_lag = 15

In [ ]:
train_X_clean, val_X_clean, test_X_clean, final_features = select_features(
    train_X=train_X,
    val_X=val_X,
    test_X=test_X,
    timeseries_features=timeseries_features,
    scores_df=scores_df,
    correlation_threshold=0.8,   # was 0.9 — catches more redundant pairs
    nzv_threshold=0.01,          # was 0.01 — increase to drop more near-zero variance
    nzv_univariate_quantile=0.5, # was 0.25 — drop bottom 50% instead of bottom 25%
)

In [ ]:
# add lagged features to train_X_clean, val_X_clean, test_X_clean
def add_lagged_features(
    df: pd.DataFrame,
    features_to_lag: list[str],
    lags: list[int],
) -> pd.DataFrame:
    lagged_cols = {
        f"{feature}_lag{lag}": df.groupby('subject_id')[feature].shift(lag)
        for feature in features_to_lag
        for lag in lags
    }
    return pd.concat([df, pd.DataFrame(lagged_cols, index=df.index)], axis=1)

def drop_lagged_nulls(df):
    lagged_cols = [col for col in df.columns if '_lag' in col]
    mask = df[lagged_cols].notna().all(axis=1)
    return df[mask].reset_index(drop=True), df.index[mask]

if max_feature_lag > 0:
    top_lag_features = get_top_features_for_lags(scores_df, final_features, top_n=20)

    train_X_clean = add_lagged_features(train_X_clean, top_lag_features, lags=list(range(1, max_feature_lag + 1)))
    val_X_clean = add_lagged_features(val_X_clean, top_lag_features, lags=list(range(1, max_feature_lag + 1)))
    test_X_clean = add_lagged_features(test_X_clean, top_lag_features, lags=list(range(1, max_feature_lag + 1)))

    train_X_clean, train_mask = drop_lagged_nulls(train_X_clean)
    train_y_clean = train_y.loc[train_mask].reset_index(drop=True)
    val_X_clean, val_mask = drop_lagged_nulls(val_X_clean)
    val_y_clean = val_y.loc[val_mask].reset_index(drop=True)
    test_X_clean, test_mask = drop_lagged_nulls(test_X_clean)
    test_y_clean = test_y.loc[test_mask].reset_index(drop=True)

In [ ]:
# Rebuild fold_indices after lagged nulls are dropped
# train_mask contains the surviving indices from original train_X
new_fold_indices = []
for tr_idx, val_idx in fold_indices:
    # Find which original indices survived the lag null drop
    surviving = set(train_mask)
    new_tr = np.array([i for i in tr_idx if i in surviving])
    new_val = np.array([i for i in val_idx if i in surviving])

    # Remap to new positional indices in train_X_clean
    old_to_new = {old: new for new, old in enumerate(train_mask)}
    new_tr  = np.array([old_to_new[i] for i in new_tr])
    new_val = np.array([old_to_new[i] for i in new_val])

    new_fold_indices.append((new_tr, new_val))

fold_indices = new_fold_indices

In [ ]:
train_X_clean.shape, val_X_clean.shape, test_X_clean.shape

In [ ]:
if with_metadata:
    cols_to_drop = ['subject_id', 'ahi_stratum', ]
else:
    cols_to_drop = [col for col in train_X_clean.columns if col in metadata_features]
    cols_to_drop += ['subject_id']

train_X_clean = train_X_clean.drop(columns=cols_to_drop, errors='ignore')
val_X_clean   = val_X_clean.drop(columns=cols_to_drop, errors='ignore')
test_X_clean  = test_X_clean.drop(columns=cols_to_drop, errors='ignore')

In [ ]:
train_X_clean.shape, val_X_clean.shape, test_X_clean.shape

Perform PCA and transform X into pc space

In [ ]:
# Perform PCA and transform X into pc space with 95% variance explained in train_X
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

if with_pca:
    scaler = StandardScaler()
    train_X_scaled = scaler.fit_transform(train_X_clean)
    pca = PCA(random_state=random_seed)
    pca.fit(train_X_scaled)
    n_components_95 = np.argmax(pca.explained_variance_ratio_.cumsum() >= 0.95) + 1

    scaler = StandardScaler()
    train_X_scaled = scaler.fit_transform(train_X_clean)
    val_X_scaled = scaler.transform(val_X_clean)
    test_X_scaled = scaler.transform(test_X_clean)

    train_X_pca = pca.transform(train_X_scaled)[:, :n_components_95]
    val_X_pca = pca.transform(val_X_scaled)[:, :n_components_95]
    test_X_pca = pca.transform(test_X_scaled)[:, :n_components_95]

In [ ]:
if with_pca:
    plt.plot(pca.explained_variance_ratio_.cumsum())
    plt.xlabel("Number of Components")
    plt.ylabel("Cumulative Explained Variance")
    plt.title("PCA Explained Variance")
    plt.show()


In [ ]:
# save data as a pickle file
import pickle

if with_pca:
    filename = f"apnea_detection_{5*max_feature_lag}secs_lag_features_{features}_pc_{5*target_future_step}secs_future_clean_metadata_{str(with_metadata).lower()}.pkl"

    with open(Path(processed_dir) / f"{filename}", "wb") as f:
        pickle.dump({
            "train_X": train_X_pca,
            "train_y": train_y,
            "fold_indices": fold_indices,
            "val_X": val_X_pca,
            "val_y": val_y,
            "test_X": test_X_pca,
            "test_y": test_y,
            "train_pca": pca,
            "train_scaler": scaler,
        }, f)

else:
    filename = f"apnea_detection_{5*max_feature_lag}secs_lag_features_{features}_{5*target_future_step}secs_future_clean_metadata_{str(with_metadata).lower()}.pkl"

    with open(Path(processed_dir) / f"{filename}", "wb") as f:
        pickle.dump({
            "train_X": train_X_clean,
            "train_y": train_y_clean,
            "fold_indices": fold_indices,
            "val_X": val_X_clean,
            "val_y": val_y_clean,
            "test_X": test_X_clean,
            "test_y": test_y_clean,
        }, f)

In [ ]:
filename